# 第13章 策略梯度方法

## 目录
1. 环境搭建
2. 策略参数化
3. REINFORCE算法
4. 编程题

---
## 1. 环境搭建

In [ ]:
!pip install numpy -q
import numpy as np
import random
print("环境搭建完成!")

---
## 2. 策略参数化

In [ ]:
def softmax_policy(theta, state):
    """Softmax策略"""
    logits = theta[state]
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / np.sum(exp_logits)

def gaussian_policy(mu, sigma):
    """高斯策略"""
    return np.random.normal(mu, sigma)

print("策略参数化: Softmax/高斯")

---
## 3. REINFORCE算法

In [ ]:
class REINFORCE:
    """REINFORCE算法"""
    def __init__(self, n_states, n_actions, alpha=0.01, gamma=0.99):
        self.n_states, self.n_actions = n_states, n_actions
        self.alpha, self.gamma = alpha, gamma
        self.theta = np.zeros((n_states, n_actions))  # 策略参数
    
    def get_policy(self, state):
        logits = self.theta[state]
        exp_logits = np.exp(logits - np.max(logits))
        return exp_logits / np.sum(exp_logits)
    
    def select_action(self, state):
        pi = self.get_policy(state)
        return np.random.choice(self.n_actions, p=pi)
    
    def update(self, trajectory, rewards):
        """策略梯度更新"""
        for t, (s, a) in enumerate(trajectory):
            G = sum(self.gamma ** k * rewards[k] for k in range(t, len(rewards)))
            pi = self.get_policy(s)
            self.theta[s, a] += self.alpha * G * (1 - pi[a])  # 简化版

def reinforce_demo():
    agent = REINFORCE(5, 2)
    for _ in range(100):
        traj = [(0, 0), (1, 1), (2, 1)]
        rews = [1.0, 1.0, 1.0]
        agent.update(traj, rews)
    print(f"策略参数: {agent.theta[0]}")

reinforce_demo()

---
## 4. 编程题

### 编程题1：实现REINFORCE算法在CartPole环境中的训练

In [ ]:
class SimpleCartPole:
    """简化CartPole环境"""
    def __init__(self):
        self.state = np.zeros(4)
        self.gravity = 9.8
        self.mass_pole = 0.1
        self.length = 0.5
        self.force = 10.0
        self.tau = 0.02
    
    def reset(self):
        self.state = np.random.randn(4) * 0.1
        return self.state.copy()
    
    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = self.force if action == 1 else -self.force
        costheta = np.cos(theta)
        sintheta = np.sin(theta)
        temp = (force + self.mass_pole * self.length * theta_dot ** 2 * sintheta) / (self.mass_pole + 0.1)
        theta_acc = (9.8 * sintheta - costheta * temp) / (self.length * (4.0/3.0 - self.mass_pole * costheta ** 2 / (self.mass_pole + 0.1)))
        x_acc = temp - self.mass_pole * theta_acc * costheta / (self.mass_pole + 0.1)
        
        x = x + self.tau * x_dot + self.tau * x_acc
        x_dot = x_dot + self.tau * x_acc
        theta = theta + self.tau * theta_dot + self.tau * theta_acc
        theta_dot = theta_dot + self.tau * theta_acc
        self.state = np.array([x, x_dot, theta, theta_dot])
        done = x < -2.4 or x > 2.4 or theta < -0.2095 or theta > 0.2095
        reward = 0 if done else 1.0
        return self.state.copy(), reward, done
    
    def get_valid_actions(self): return [0, 1]

class REINFORCEAgent:
    """REINFORCE智能体"""
    def __init__(self, n_states, n_actions, alpha=0.001, gamma=0.99):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.theta = np.random.randn(n_states, n_actions) * 0.01
    
    def get_probs(self, state):
        s = int(np.clip(state[0] * 3, -4, 4) + 4) % self.n_states
        logits = self.theta[s]
        exp_logits = np.exp(logits - np.max(logits))
        return exp_logits / np.sum(exp_logits)
    
    def select_action(self, state):
        probs = self.get_probs(state)
        return np.random.choice(self.n_actions, p=probs)
    
    def update(self, trajectory, rewards):
        G = 0
        for t in range(len(trajectory) - 1, -1, -1):
            G = self.gamma * G + rewards[t]
            s, a = trajectory[t]
            s_idx = int(np.clip(s[0] * 3, -4, 4) + 4) % self.n_states
            probs = self.theta[s_idx]
            exp_probs = np.exp(probs - np.max(probs))
            probs = exp_probs / np.sum(exp_probs)
            self.theta[s_idx, a] += self.alpha * G * (1 - probs[a])

def train_reinforce():
    """训练REINFORCE"""
    n_states, n_actions = 8, 2
    agent = REINFORCEAgent(n_states, n_actions)
    rewards_history = []
    
    for ep in range(200):
        env = SimpleCartPole()
        s = env.reset()
        trajectory, rewards = [], []
        total_reward = 0
        
        for step in range(500):
            a = agent.select_action(s)
            ns, r, done = env.step(a)
            trajectory.append(s.copy())
            rewards.append(r)
            s = ns
            total_reward += r
            if done: break
        
        rewards_history.append(total_reward)
        agent.update(trajectory, rewards)
    
    print(f"REINFORCE训练结果:")
    print(f"  最终回报: {rewards_history[-1]}")
    print(f"  平均回报(最后10局): {np.mean(rewards_history[-10:]):.1f}")

train_reinforce()

---

### 编程题2：实现Actor-Critic算法并与REINFORCE进行对比

In [ ]:
class ActorCritic:
    """Actor-Critic算法"""
    def __init__(self, n_states, n_actions, alpha_a=0.001, alpha_c=0.01, gamma=0.99):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha_a = alpha_a
        self.alpha_c = alpha_c
        self.gamma = gamma
        self.theta = np.zeros((n_states, n_actions))  # Actor
        self.w = np.zeros(n_states)  # Critic
    
    def get_policy(self, state):
        logits = self.theta[state]
        exp_logits = np.exp(logits - np.max(logits))
        return exp_logits / np.sum(exp_logits)
    
    def get_value(self, state):
        return self.w[state]
    
    def select_action(self, state):
        probs = self.get_policy(state)
        return np.random.choice(self.n_actions, p=probs)
    
    def update(self, s, a, r, ns, done):
        target = r if done else r + self.gamma * self.get_value(ns)
        td_error = target - self.get_value(s)
        self.w[s] += self.alpha_c * td_error
        probs = self.get_policy(s)
        self.theta[s, a] += self.alpha_a * td_error * (1 - probs[a])

def test_actor_critic():
    """测试Actor-Critic"""
    n_states, n_actions = 5, 2
    ac = ActorCritic(n_states, n_actions)
    rewards_history = []
    
    class ACEnv:
        def __init__(self):
            self.state = 2
        def reset(self): self.state = 2; return self.state
        def step(self, a):
            if a == 1: self.state = min(4, self.state + 1)
            else: self.state = max(0, self.state - 1)
            r = 1.0 if self.state == 4 else -0.01
            done = self.state == 4
            return self.state, r, done
    
    env = ACEnv()
    for ep in range(200):
        s = env.reset()
        total_reward = 0
        for _ in range(50):
            a = ac.select_action(s)
            ns, r, done = env.step(a)
            ac.update(s, a, r, ns, done)
            s = ns
            total_reward += r
            if done: break
        rewards_history.append(total_reward)
    
    print(f"Actor-Critic训练:")
    print(f"  最终回报: {rewards_history[-1]}")
    print(f"  收敛步数: {next((i for i, r in enumerate(rewards_history) if r > 0.9), -1)}")

test_actor_critic()

---

### 编程题3：实现连续动作空间的策略梯度(高斯策略)

In [ ]:
class GaussianPolicy:
    """高斯策略(连续动作)"""
    def __init__(self, n_states, alpha=0.01, gamma=0.99):
        self.n_states = n_states
        self.alpha = alpha
        self.gamma = gamma
        self.theta = np.zeros(n_states)  # 均值参数
        self.log_std = np.zeros(n_states)  # log标准差
    
    def get_action(self, state):
        std = np.exp(self.log_std[state])
        action = self.theta[state] + np.random.randn() * std
        return np.clip(action, -1, 1)
    
    def get_log_prob(self, state, action):
        std = np.exp(self.log_std[state])
        diff = action - self.theta[state]
        return -0.5 * (diff / std) ** 2 - np.log(std) - 0.5 * np.log(2 * np.pi)
    
    def update(self, trajectory, advantages):
        for (s, a), adv in zip(trajectory, advantages):
            std = np.exp(self.log_std[s])
            diff = a - self.theta[s]
            self.theta[s] += self.alpha * adv * diff / std
            self.log_std[s] += self.alpha * adv * (diff ** 2 / std - 1) * 0.5

class ContinuousEnv:
    """连续动作环境"""
    def __init__(self):
        self.state = 0.0
    def reset(self): self.state = 0.0; return self.state
    def step(self, a):
        self.state += a * 0.1 + np.random.randn() * 0.05
        r = -abs(self.state - 1.0)
        done = abs(self.state - 1.0) < 0.1
        return self.state, r, done

def test_continuous_policy():
    """测试连续动作策略"""
    n_states = 1
    policy = GaussianPolicy(n_states)
    rewards_history = []
    
    env = ContinuousEnv()
    for ep in range(200):
        s = env.reset()
        trajectory, rewards = [], []
        total_reward = 0
        
        for _ in range(100):
            a = policy.get_action(s)
            ns, r, done = env.step(a)
            trajectory.append((int(s), a))
            rewards.append(r)
            s = ns
            total_reward += r
            if done: break
        
        rewards_history.append(total_reward)
        advantages = []
        G = 0
        for t in range(len(rewards) - 1, -1, -1):
            G = policy.gamma * G + rewards[t]
            advantages.insert(0, G)
        policy.update(trajectory, advantages)
    
    print("高斯策略梯度训练:")
    print(f"  最终回报: {rewards_history[-1]:.2f}")
    print(f"  学到的均值(状态0): {policy.theta[0]:.2f}")

test_continuous_policy()

print("\n第13章 策略梯度方法 - 编程题完成!")